In [259]:
import importlib
import tau
from math import gcd
import numpy as np
from math import floor
from math import ceil
from fractions import Fraction
#importlib.reload(tau)
from iteration_utilities import deepflatten
from itertools import combinations
from itertools import product


In [208]:
def all_coprime(p,q,r):
    return(gcd(q,r)==1  & gcd(p,q)==1 & gcd(p,r)==1)


def hirzebruch_cont_fr(num, den):
    f = Fraction(num, den)
    res = []
    
    while True:
        # 1. Take the ceiling instead of the floor
        # math.ceil returns an int, which is perfect for our list
        b = ceil(f.numerator / f.denominator)
        res.append(b)
        
        # 2. Calculate the remainder: (b - f)
        remainder = b - f
        
        # 3. If the remainder is 0, we are done
        if remainder == 0:
            break
            
        # 4. Take the reciprocal for the next iteration
        f = 1 / remainder
        
    return res

def inverse_hirzebruch(cf):
    if not cf:
        return None # Or raise an error
    
    # Start with the last element
    res = Fraction(cf[-1], 1)
    
    # Work backwards through the list (excluding the last element)
    for b in reversed(cf[:-1]):
        # The formula is: b - (1 / previous_result)
        res = b - (1 / res)
        
    return res.numerator, res.denominator

def plumbing_weights(p,q,r):
    x = (-pow(q * r, -1, p)) % p
    y = (-pow(r * p, -1, q)) % q
    z = (-pow(p * q, -1, r)) % r
    e = (x*q*r+y*r*p+z*p*q+1) // (p*q*r)
    return e, hirzebruch_cont_fr(p,x),hirzebruch_cont_fr(q,y),hirzebruch_cont_fr(r,z)

def plumbing_matrix(data):
    # Unpack the input: (central, leg1, leg2, leg3)
    central_val, leg1, leg2, leg3 = data
    legs = [leg1, leg2, leg3]
    
    # 1. Calculate total number of vertices
    # Total = 1 (center) + sum of lengths of all legs
    total_vertices = 1 + sum(len(leg) for leg in legs)
    
    # Initialize a square matrix with zeros
    matrix = np.zeros((total_vertices, total_vertices), dtype=int)
    
    # 2. Set the central vertex (Index 0)
    # Decorations are negated per your instructions
    matrix[0, 0] = -central_val
    
    current_idx = 1
    for leg in legs:
        for i, val in enumerate(leg):
            # Set the diagonal decoration (negated)
            matrix[current_idx, current_idx] = -val
            
            if i == 0:
                # Connect the first node of the leg to the central vertex (0)
                matrix[0, current_idx] = 1
                matrix[current_idx, 0] = 1
            else:
                # Connect to the previous node in the same leg
                matrix[current_idx - 1, current_idx] = 1
                matrix[current_idx, current_idx - 1] = 1
            
            current_idx += 1
            
    return matrix

def list_coprime_triples(N):
    for p, q, r in combinations(range(2, N), 3):
        if gcd(p, q) == 1 and gcd(q, r) == 1 and gcd(p, r) == 1:
            yield p, q, r

def all_even(numbers):
    return all(x % 2 == 0 for x in numbers)

def spec(p,q,r):
    spectrum = []
    for i in range(1,p):
        for j in range(1,q):
            for k in range(1,r):
                spectrum.append(Fraction(i,p)+Fraction(j,q)+Fraction(k,r))
    return spectrum

def pg(p,q,r):
    return len([x for x in spec(p,q,r) if x<1])

def casson(p,q,r):
    return (2*len([x for x in spec(p,q,r) if x<1]) - len([x for x in spec(p,q,r) if 1<x<2]))//8
                

In [4]:

def find_last_increase_index(arr):
    """
    Returns the index of the last element that was greater than its predecessor.
    Returns None if no increase is found.
    """
    # Convert to numpy array if it isn't one already
    arr = np.asanyarray(arr)
    
    # Calculate differences: diff[i] = arr[i+1] - arr[i]
    diffs = np.diff(arr)
    
    # Find all indices where the difference is positive
    increase_indices = np.where(diffs > 0)[0]
    
    if increase_indices.size > 0:
        # np.diff shifts indices by 1, so the increase ends at index + 1
        return increase_indices[-1] + 1
    
    return None

def find_first_two(arr):
    # np.where returns indices where the condition (arr == 2) is True
    indices = np.where(arr == 2)[0]
    
    if indices.size > 0:
        return indices[0]  # Return the first found index
    return -1

def find_representation(n, q, r):
    # assumes gcd(q, r) == 1

    # modular inverse of q mod r
    def egcd(a, b):
        if b == 0:
            return a, 1, 0
        g, x, y = egcd(b, a % b)
        return g, y, x - (a // b) * y

    def modinv(a, m):
        _, x, _ = egcd(a, m)
        return x % m

    inv_q = modinv(q % r, r)

    # initial solution for a modulo r
    a = (n % r) * inv_q % r
    b = (n - a * q) // r

    if b < 0:
        # shift by full periods of r
        k = (a * q - n + r - 1) // (q * r)
        a -= k * r
        b = (n - a * q) // r

    if a < 0 or b < 0:
        return None

    return a, b


In [4]:
for q in range(3,120):
    for r in range(6,120):
        if (r<2*q) & (3*q<2*r) & all_coprime(2,q,r):
            n0=2*q*r-q*r-2*q-2*r
            ks = tau.tau_sequence(2,q,r)[1]
            differences= np.diff(tau.tau_sequence(2,q,r)[-1] //2)
            ones = np.count_nonzero(differences==1)
            minusones = np.count_nonzero(differences==-1)
            print(q,r,ks//2+ones-minusones)
            #b = int(b)  

5 9 0
7 11 -1
7 13 -1
9 17 -3
11 17 -5
11 19 -7
11 21 -6
13 21 -10
13 23 -10
13 25 -10
15 23 -12
15 29 -15
17 27 -18
17 29 -20
17 31 -22
17 33 -21
19 29 -22
19 31 -25
19 33 -28
19 35 -28
19 37 -28
21 37 -34
21 41 -36
23 35 -35
23 37 -40
23 39 -42
23 41 -44
23 43 -46
23 45 -45
25 39 -46
25 41 -49
25 43 -52
25 47 -55
25 49 -55
27 41 -51
27 43 -55
27 47 -63
27 49 -64
27 53 -66
29 45 -63
29 47 -68
29 49 -70
29 51 -72
29 53 -77
29 55 -79
29 57 -78
31 47 -70
31 49 -76
31 51 -79
31 53 -82
31 55 -85
31 57 -88
31 59 -91
31 61 -91
33 53 -90
33 59 -99
33 61 -103
33 65 -105
35 53 -92
35 57 -102
35 59 -107
35 61 -112
35 67 -121
35 69 -120
37 57 -109
37 59 -112
37 61 -118
37 63 -121
37 65 -124
37 67 -130
37 69 -133
37 71 -136
37 73 -136
39 59 -117
39 61 -124
39 67 -139
39 71 -147
39 73 -151
39 77 -153
41 63 -135
41 65 -140
41 67 -145
41 69 -150
41 71 -155
41 73 -160
41 75 -165
41 77 -167
41 79 -172
41 81 -171
43 65 -145
43 67 -154
43 69 -160
43 71 -163
43 73 -169
43 75 -175
43 77 -178
43 79 -181
43 

In [109]:
for a in list_coprime_triples(100):
    p,q,r = a
    weights = list(deepflatten(plumbing_weights(p,q,r)))
    if all_even(weights):
        #print(p,q,r,f"   d = {tau.tau_sequence(p,q,r)[0]}")
        if tau.tau_sequence(p,q,r)[0]!=len(list(deepflatten(plumbing_weights(p,q,r))))//4:
            print(p,q,r)
        #print(f"    casson = {-casson(p,q,r)}")
        #print(f"        pg = {pg(p,q,r)}")

In [51]:
1/2<7/11<2/3

True

In [302]:
entries = [0, 1,-1]
    

for a in list_coprime_triples(100):
    p,q,r = a
    if (p-1)/p<q/r<p/(p+1):
        weight_list = list(deepflatten(plumbing_weights(p,q,r)))
        s = len(weight_list)
        inv = np.linalg.inv(plumbing_matrix(plumbing_weights(p,q,r)))
        if s < 20:
            t = tau.tau_sequence(p,q,r)[0]
            print(p,q,r)
            print(plumbing_weights(p,q,r))
            count = 0
            for vect in product(entries, repeat=s):
                k = np.array(vect)
                k2 = round(((k @ inv @ k) + s)/4)
                if t == k2:
                    k_min = k
                    count = count + 1
                    if count==2:
                        break
            if count == 0:
                print("no vectors found")
                print(p,q,r)

2 3 5
(2, [2], [2, 2], [2, 2, 2, 2])
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 1]
2 5 9
(2, [2], [2, 3], [2, 2, 2, 2, 2, 2, 2, 2])
[0 0 0 0 0 0 0 0 0 0 1 0]
[ 0  0  0  0  0  0  0  0  0  0 -1  0]
2 7 11
(2, [2], [2, 2, 2, 2, 2, 2], [2, 3, 2, 2])
[0 0 0 0 0 0 0 0 0 0 1 0]
[ 0  0  0  0  0  0  0  0  0  0 -1  0]
2 7 13
(2, [2], [2, 4], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
2 11 17
(2, [2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 4, 2, 2])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
2 11 19
(1, [2], [6, 2], [4, 2, 2, 2, 2, 2])
[0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1]
2 13 21
(1, [2], [4, 2, 2, 2], [6, 2, 2, 2])
[0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1]
2 13 23
(2, [2], [2, 2, 2, 2, 2, 3], [2, 3, 2, 2, 2, 2, 2, 2])
[ 0  0  0  0  0  0  0  0  0  0  0  0  1 -1 -1 -1]
[ 0  0  0  0  0  0  0  0  0  0  0  0 -1  1  1  1]
2 17 27
(2, [2], [2, 3, 2, 2, 2, 2], [2, 2, 2, 2, 2, 3, 2, 2])
[ 0  0  0  0  0  0  0  0  0  0  0  

In [304]:
tau.tau_sequence(2,7,11)

(2,
 0,
 array([ 0, -2, -2, -2, -2, -2,  0,  0,  0,  0,  0,  0,  0,  0,  2,  0,  0,
         0,  0,  0,  2,  2,  2,  0,  0,  0,  0,  0,  2,  0,  0,  0,  0,  0,
         0,  0,  0, -2, -2, -2, -2, -2]))

milnor = 60, p_g = 5, casson = sign/8 = 5-50+5=-5, k^2+s = 12p_g - milnor = 12*5-60=0, number of x<N_0/pqr: b_3+b_5 Instanton floer
rank HF_red = d/2 - casson = 6?

In [ ]:
44,55,66,77,88

In [328]:
np.where(tau.tau_sequence(5,8,17)[2]>=2)

(array([199, 200, 204, 205, 210, 215, 216, 220, 221]),)

In [202]:
for n in range(6):
    q,r = inverse_hirzebruch([1,3,2,2*n,2])
    print(tau.tau_sequence(2,q,r)[0],len(list(deepflatten(plumbing_weights(2,q,r))))//8)
    print(plumbing_weights(2,q,r))
    

2 1
(2, [2], [2, 2, 2, 2, 2, 2], [2, 3, 2, 2])
2 1
(2, [2], [2, 3], [2, 2, 2, 2, 2, 2, 2, 2])
2 1
(2, [2], [2, 2, 4, 2], [2, 2, 2, 3, 2, 2, 2, 2])
2 1
(2, [2], [2, 2, 3, 3, 2], [2, 2, 2, 4, 2, 2, 2, 2])
2 2
(2, [2], [2, 2, 3, 2, 3, 2], [2, 2, 2, 5, 2, 2, 2, 2])
2 2
(2, [2], [2, 2, 3, 2, 2, 3, 2], [2, 2, 2, 6, 2, 2, 2, 2])


In [206]:
for a in list_coprime_triples(50):
    p,q,r = a
    if p==2:
        print(q,r)
        print(plumbing_weights(p,q,r))

3 5
(2, [2], [2, 2], [2, 2, 2, 2])
3 7
(1, [2], [3], [7])
3 11
(2, [2], [2, 2], [2, 2, 2, 2, 3])
3 13
(1, [2], [3], [7, 2])
3 17
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2])
3 19
(1, [2], [3], [7, 2, 2])
3 23
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2, 2])
3 25
(1, [2], [3], [7, 2, 2, 2])
3 29
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2, 2, 2])
3 31
(1, [2], [3], [7, 2, 2, 2, 2])
3 35
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2, 2, 2, 2])
3 37
(1, [2], [3], [7, 2, 2, 2, 2, 2])
3 41
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2, 2, 2, 2, 2])
3 43
(1, [2], [3], [7, 2, 2, 2, 2, 2, 2])
3 47
(2, [2], [2, 2], [2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2])
3 49
(1, [2], [3], [7, 2, 2, 2, 2, 2, 2, 2])
5 7
(1, [2], [5], [4, 2])
5 9
(2, [2], [2, 3], [2, 2, 2, 2, 2, 2, 2, 2])
5 11
(1, [2], [3, 2], [11])
5 13
(2, [2], [2, 2, 2, 2], [2, 2, 5])
5 17
(1, [2], [5], [4, 2, 3])
5 19
(2, [2], [2, 3], [2, 2, 2, 2, 2, 2, 2, 2, 3])
5 21
(1, [2], [3, 2], [11, 2])
5 23
(2, [2], [2, 2, 2, 2], [2, 2, 5, 2])
5 27
(1, [2], [5], [4, 2, 3, 2])
5 29
(2, [2], [2, 3], [2, 2

In [310]:
tau.tau_sequence(2,7,13)==2

False

In [329]:
for a in list_coprime_triples(100):
    p,q,r = a
    if (p-1)/p<q/r<p/(p+1):
        print(p,q,r,np.where(tau.tau_sequence(p,q,r)[2]>=2))

2 3 5 (array([0]),)
2 5 9 (array([ 0,  8,  9, 10]),)
2 7 11 (array([14, 20, 21, 22, 28]),)
2 7 13 (array([12, 13, 14, 24, 25, 26, 27, 28, 38, 39, 40]),)
2 9 17 (array([32, 33, 34, 35, 36, 48, 49, 50, 51, 52, 53, 54, 66, 67, 68, 69, 70]),)
2 11 17 (array([44, 54, 55, 56, 64, 65, 66, 67, 68, 76, 77, 78, 88]),)
2 11 19 (array([74, 75, 76]),)
2 11 21 (array([ 42,  60,  61,  62,  63,  64,  65,  66,  80,  81,  82,  83,  84,
        85,  86,  87,  88, 102, 103, 104, 105, 106, 107, 108, 126]),)
2 13 21 (array([102, 103, 104]),)
2 13 23 (array([ 90,  91,  92,  98, 104, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 124, 130, 136, 137, 138]),)
2 13 25 (array([ 74,  75,  76,  96,  97,  98,  99, 100, 101, 102, 103, 104, 120,
       121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 146, 147, 148,
       149, 150, 151, 152, 153, 154, 174, 175, 176]),)
2 15 23 (array([ 90, 104, 105, 106, 118, 119, 120, 121, 122, 132, 133, 134, 135,
       136, 137, 138, 148, 149, 150, 151, 152, 164, 165, 166, 18